# 03 — San Francisco: Data Exploration

**Milestone 1** — pulling a live sample from DataSF to verify the real schema before we design the cross-city pipeline (Milestone 2).

- **Dataset:** "Police Department Incident Reports: 2018 to Present", DataSF (Socrata/SODA API)
- **Dataset ID:** `wg3w-h783`
- **Scope of this notebook:** a recent-window sample; the full pull (still much smaller than Chicago/NYC's full history) happens properly in the Milestone 2 pipeline.

See `docs/DATA_SOURCES.md` for background and `docs/ETHICS.md` for how this data may and may not be used.

In [1]:
import requests
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
DATASET_ENDPOINT = "https://data.sfgov.org/resource/wg3w-h783.json"
SAMPLE_SIZE = 20_000
PAGE_SIZE = 5_000

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def fetch_socrata_sample(endpoint: str, total: int, page_size: int, order: str = ":id") -> pd.DataFrame:
    """Pull `total` rows from a Socrata endpoint via offset pagination, most-recent-first."""
    frames = []
    fetched = 0
    while fetched < total:
        limit = min(page_size, total - fetched)
        params = {
            "$limit": limit,
            "$offset": fetched,
            "$order": f"{order} DESC",
        }
        response = requests.get(endpoint, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()
        if not page:
            break
        frames.append(pd.DataFrame(page))
        fetched += len(page)
        if len(page) < limit:
            break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


sf_df = fetch_socrata_sample(DATASET_ENDPOINT, SAMPLE_SIZE, PAGE_SIZE, order="incident_datetime")
sf_df.shape

(20000, 29)

In [4]:
sf_df.dtypes

row_id                      object
incident_datetime           object
incident_date               object
incident_time               object
incident_year               object
incident_day_of_week        object
report_datetime             object
incident_id                 object
incident_number             object
cad_number                  object
report_type_code            object
report_type_description     object
incident_code               object
incident_category           object
incident_subcategory        object
incident_description        object
resolution                  object
police_district             object
data_as_of                  object
data_loaded_at              object
intersection                object
cnn                         object
analysis_neighborhood       object
supervisor_district         object
supervisor_district_2012    object
latitude                    object
longitude                   object
point                       object
filed_online        

In [5]:
sf_df.head(3)

,row_id,incident_datetime,incident_date,incident_time,incident_year,incident_day_of_week,report_datetime,incident_id,incident_number,cad_number,report_type_code,report_type_description,incident_code,incident_category,incident_subcategory,incident_description,resolution,police_district,data_as_of,data_loaded_at,intersection,cnn,analysis_neighborhood,supervisor_district,supervisor_district_2012,latitude,longitude,point,filed_online
0,158072562071,2026-07-01T23:14:00.000,2026-07-01T00:00:00.000,23:14,2026,Wednesday,2026-07-01T23:14:00.000,1580725,260372560,261823711,II,Initial,62071,Warrant,Other,Probation Search,Cite or Arrest Adult,Central,2026-07-02T09:34:07.000,2026-07-03T09:57:13.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,158072516220,2026-07-01T23:14:00.000,2026-07-01T00:00:00.000,23:14,2026,Wednesday,2026-07-01T23:14:00.000,1580725,260372560,261823711,II,Initial,16220,Drug Offense,Drug Violation,"Opiates, Possession For Sale",Cite or Arrest Adult,Central,2026-07-02T09:34:07.000,2026-07-03T09:57:13.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,158072526170,2026-07-01T23:14:00.000,2026-07-01T00:00:00.000,23:14,2026,Wednesday,2026-07-01T23:14:00.000,1580725,260372560,261823711,II,Initial,26170,Other Miscellaneous,Other,Probation Violation,Cite or Arrest Adult,Central,2026-07-02T09:34:07.000,2026-07-03T09:57:13.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data quality checks

In [6]:
null_rates = (sf_df.isna().mean() * 100).round(2).sort_values(ascending=False)
null_rates[null_rates > 0]

filed_online                89.99
cad_number                  12.62
supervisor_district          3.44
supervisor_district_2012     3.41
analysis_neighborhood        3.41
point                        3.40
longitude                    3.40
latitude                     3.40
cnn                          3.40
intersection                 3.40
incident_category            0.43
incident_subcategory         0.43
dtype: float64

In [7]:
sf_df["incident_datetime"] = pd.to_datetime(sf_df["incident_datetime"])
print("date range in this sample:", sf_df["incident_datetime"].min(), "to", sf_df["incident_datetime"].max())

date range in this sample: 2026-04-09 02:30:00 to 2026-07-01 23:14:00


In [8]:
sf_df["incident_category"].value_counts().head(20)

incident_category
Larceny Theft                               3366
Drug Offense                                2435
Assault                                     1641
Warrant                                     1590
Other Miscellaneous                         1514
Malicious Mischief                          1058
Non-Criminal                                 894
Burglary                                     748
Motor Vehicle Theft                          735
Disorderly Conduct                           563
Lost Property                                560
Missing Person                               512
Fraud                                        512
Recovered Vehicle                            462
Miscellaneous Investigation                  454
Suspicious Occ                               440
Robbery                                      356
Offences Against The Family And Children     309
Traffic Violation Arrest                     302
Traffic Collision                            237
Na

In [9]:
sf_df["resolution"].value_counts(dropna=False)

resolution
Open or Active          11894
Cite or Arrest Adult     8002
Unfounded                  84
Exceptional Adult          20
Name: count, dtype: int64

In [10]:
# analysis_neighborhood is SF's own official neighborhood boundary layer -- already a named neighborhood field,
# unlike Chicago (numeric code needing a join) or NYC (borough/precinct only, needs a spatial join). Best of the three for mapping.
print("unique neighborhoods:", sf_df["analysis_neighborhood"].nunique())
print("rows missing neighborhood:", sf_df["analysis_neighborhood"].isna().sum())
sf_df["analysis_neighborhood"].value_counts().head(10)

unique neighborhoods: 41
rows missing neighborhood: 682


analysis_neighborhood
Mission                           3334
Tenderloin                        2999
South of Market                   2619
Bayview Hunters Point             1351
Financial District/South Beach     773
Western Addition                   541
Nob Hill                           506
Castro/Upper Market                478
Hayes Valley                       412
Mission Bay                        392
Name: count, dtype: int64

In [11]:
sample_path = RAW_DATA_DIR / "sf_sample.csv"
sf_df.to_csv(sample_path, index=False)
print(f"saved {len(sf_df):,} rows to {sample_path}")

saved 20,000 rows to ../data/raw/sf_sample.csv


## Findings

- **Sample:** 20,000 rows, most-recent-first, spans 2026-04-09 to 2026-07-01 — about **three months**, versus roughly one month for the same row count in Chicago and two weeks in NYC. SF's reporting volume is meaningfully smaller than the other two cities, which is worth stating plainly in the site's narrative (smaller city, smaller absolute numbers — comparisons across cities should favor rates, not raw counts, for exactly this reason).

- **Null rates:** `filed_online` (90.0%) is mostly unusable/sparse. `cad_number` (12.6%) is a computer-aided-dispatch linkage field, moderately incomplete. The geography cluster (`analysis_neighborhood`, `latitude`, `longitude`, `point`, `cnn`, `intersection`, `supervisor_district*`) is all missing together at ~3.4% — the same underlying rows lack geocoding, same pattern as Chicago. `incident_category`/`incident_subcategory` missing at 0.43% — negligible.

- **Top incident categories (`incident_category`):** Larceny Theft (3,366), Drug Offense (2,435), Assault (1,641), Warrant (1,590) lead. Notably present: "Offences Against The Family And Children" (309) — SF's closest equivalent to Chicago's `domestic` flag, but it's a category label, not a boolean flag, so it's coarser and likely undercounts domestic-related incidents that get filed under Assault or other categories instead.

- **Resolution split:** 59.5% "Open or Active", 40.0% "Cite or Arrest Adult", with small "Unfounded" (84) and "Exceptional Adult" (20) categories. Useful outcome field, but "Open or Active" dominating means most incidents in this recent window haven't resolved yet — a real censoring issue if we ever want to analyze resolution *time*, not just resolution *type*.

- **Neighborhood field is the best of the three cities:** `analysis_neighborhood` gives 41 named neighborhoods directly (Mission, Tenderloin, South of Market lead by volume) with no code-to-name join required, unlike Chicago's numeric `community_area` or NYC's borough/precinct-only geography. Only 3.41% missing (682 rows). This makes SF the easiest city to prototype the mapping/neighborhood-aggregation pipeline against before tackling Chicago and NYC's join logic in Milestone 2.

- **Nothing here changes `docs/ETHICS.md`** — no individual-level demographic fields in this dataset either (confirmed: category, resolution, geography, and IDs only). The ethics policy on demographic data applies specifically to NYC.